# Forecasting de ventas 2025

In [2]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as sk
import streamlit as st
import holidays

In [3]:
from pathlib import Path

base_dir = Path.cwd()
if base_dir.name != "notebooks":
    base_dir = base_dir / "notebooks"

inferencia_path = base_dir.parent / "data" / "raw" / "inferencia" / "ventas_2025_inferencia.csv"
inferencia_df = pd.read_csv(inferencia_path)

In [4]:
# Sección de transformación de datos
inferencia_df["fecha"] = pd.to_datetime(inferencia_df["fecha"], errors="coerce")
inferencia_df = inferencia_df.sort_values(["fecha", "producto_id"]).reset_index(drop=True)

# Variables temporales y de calendario
inferencia_df["anio"] = inferencia_df["fecha"].dt.year
inferencia_df["mes"] = inferencia_df["fecha"].dt.month
inferencia_df["dia_mes"] = inferencia_df["fecha"].dt.day
inferencia_df["dia_semana"] = inferencia_df["fecha"].dt.day_name()
inferencia_df["dia_semana_num"] = inferencia_df["fecha"].dt.dayofweek
inferencia_df["dia_semana_ordenado"] = pd.Categorical(
    inferencia_df["dia_semana"],
    categories=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
    ordered=True,
)
inferencia_df["dia_del_año"] = inferencia_df["fecha"].dt.dayofyear
inferencia_df["semana_del_año"] = inferencia_df["fecha"].dt.isocalendar().week.astype(int)
inferencia_df["trimestre"] = inferencia_df["fecha"].dt.quarter
inferencia_df["cuatrimestre"] = ((inferencia_df["mes"] - 1) // 4) + 1
inferencia_df["mes_nombre"] = inferencia_df["fecha"].dt.month_name()
inferencia_df["es_fin_de_semana"] = inferencia_df["dia_semana_num"].isin([5, 6])
inferencia_df["es_inicio_mes"] = inferencia_df["fecha"].dt.is_month_start
inferencia_df["es_fin_mes"] = inferencia_df["fecha"].dt.is_month_end
inferencia_df["es_inicio_trimestre"] = inferencia_df["fecha"].dt.is_quarter_start
inferencia_df["es_fin_trimestre"] = inferencia_df["fecha"].dt.is_quarter_end
inferencia_df["es_primera_quincena"] = inferencia_df["dia_mes"].le(15)
inferencia_df["es_segunda_quincena"] = inferencia_df["dia_mes"].gt(15)

# Estacionalidad adicional
inferencia_df["estacion"] = inferencia_df["mes"].map({
    12: 1, 1: 1, 2: 1,
    3: 2, 4: 2, 5: 2,
    6: 3, 7: 3, 8: 3,
    9: 4, 10: 4, 11: 4,
})
inferencia_df["es_verano"] = inferencia_df["estacion"].eq(3).astype(int)
inferencia_df["es_nuevo_año"] = ((inferencia_df["mes"] == 1) & (inferencia_df["dia_mes"] == 1)).astype(int)

# Festivos y eventos comerciales
years = sorted(inferencia_df["anio"].dropna().astype(int).unique())
holiday_calendar = holidays.country_holidays("ES", years=years)
inferencia_df["es_festivo"] = inferencia_df["fecha"].isin(list(holiday_calendar.keys()))
inferencia_df["nombre_festivo"] = inferencia_df["fecha"].apply(lambda x: holiday_calendar.get(x, "") if pd.notna(x) else "")

black_friday_dates = [
    pd.Timestamp(year, 11, 1) + pd.offsets.WeekOfMonth(week=3, weekday=3) + pd.Timedelta(days=1)
    for year in years
]
cyber_monday_dates = [
    pd.Timestamp(year, 11, 1) + pd.offsets.WeekOfMonth(week=3, weekday=3) + pd.Timedelta(days=8)
    for year in years
]
inferencia_df["es_black_friday"] = inferencia_df["fecha"].isin(black_friday_dates)
inferencia_df["es_ciber_monday"] = inferencia_df["fecha"].isin(cyber_monday_dates)
inferencia_df["es_navidad"] = inferencia_df["mes"].eq(12) & inferencia_df["dia_mes"].isin([24, 25, 26, 31])
inferencia_df["es_reyes"] = inferencia_df["mes"].eq(1) & inferencia_df["dia_mes"].eq(6)
inferencia_df["es_san_juan"] = inferencia_df["mes"].eq(6) & inferencia_df["dia_mes"].eq(24)
inferencia_df["es_ha_duplicado"] = inferencia_df.duplicated(subset=["fecha", "producto_id"], keep=False)

# Conversión de precios
for col in ["precio_base", "precio_venta", "Amazon", "Decathlon", "Deporvillage"]:
    if col in inferencia_df.columns:
        inferencia_df[col] = pd.to_numeric(inferencia_df[col], errors="coerce")

# Creación de lags y media móvil
if {"anio", "fecha", "producto_id", "unidades_vendidas"}.issubset(inferencia_df.columns):
    inferencia_df = inferencia_df.sort_values(["anio", "fecha", "producto_id"]).reset_index(drop=True)
    inferencia_df["lag_1"] = inferencia_df.groupby("anio")["unidades_vendidas"].shift(1)
    inferencia_df["lag_2"] = inferencia_df.groupby("anio")["unidades_vendidas"].shift(2)
    inferencia_df["lag_3"] = inferencia_df.groupby("anio")["unidades_vendidas"].shift(3)
    inferencia_df["lag_4"] = inferencia_df.groupby("anio")["unidades_vendidas"].shift(4)
    inferencia_df["lag_5"] = inferencia_df.groupby("anio")["unidades_vendidas"].shift(5)
    inferencia_df["lag_6"] = inferencia_df.groupby("anio")["unidades_vendidas"].shift(6)
    inferencia_df["lag_7"] = inferencia_df.groupby("anio")["unidades_vendidas"].shift(7)
    inferencia_df["rolling_mean_7"] = (
        inferencia_df.groupby("anio")["unidades_vendidas"]
        .transform(lambda s: s.shift(1).rolling(window=7, min_periods=7).mean())
    )

# Descuento y precios de competencia
if {"precio_venta", "precio_base"}.issubset(inferencia_df.columns):
    inferencia_df["descuento_pct"] = ((inferencia_df["precio_base"] - inferencia_df["precio_venta"]) / inferencia_df["precio_base"]) * 100
    inferencia_df["descuento_porcentaje"] = ((inferencia_df["precio_venta"] - inferencia_df["precio_base"]) / inferencia_df["precio_base"]) * 100

competidores = ["Amazon", "Decathlon", "Deporvillage"]
if all(col in inferencia_df.columns for col in competidores):
    inferencia_df["precio_competencia"] = inferencia_df[competidores].mean(axis=1)
    inferencia_df["ratio_precio"] = inferencia_df["precio_venta"] / inferencia_df["precio_competencia"]
    inferencia_df = inferencia_df.drop(columns=competidores)

# One-hot encoding y alineación de columnas con el DataFrame histórico
if {"nombre", "categoria", "subcategoria"}.issubset(inferencia_df.columns):
    inferencia_df["nombre_h"] = inferencia_df["nombre"]
    inferencia_df["categoria_h"] = inferencia_df["categoria"]
    inferencia_df["subcategoria_h"] = inferencia_df["subcategoria"]
    inferencia_df = pd.get_dummies(
        inferencia_df,
        columns=["nombre_h", "categoria_h", "subcategoria_h"],
        prefix_sep="_",
        drop_first=False,
    )

training_df_path = base_dir.parent / "data" / "processed" / "df.csv"
if training_df_path.exists():
    target_columns = pd.read_csv(training_df_path, nrows=0).columns.tolist()
    missing_columns = [col for col in target_columns if col not in inferencia_df.columns]
    for col in missing_columns:
        inferencia_df[col] = 0
    extra_columns = [col for col in inferencia_df.columns if col not in target_columns]
    if extra_columns:
        inferencia_df = inferencia_df.drop(columns=extra_columns)
    inferencia_df = inferencia_df[target_columns]
else:
    raise FileNotFoundError(f"No se encontró el DataFrame histórico procesado en {training_df_path}")

print("Registros totales tras preprocesamiento:", len(inferencia_df))
print("Registros octubre:", (inferencia_df["mes"] == 10).sum())
print("Registros noviembre:", (inferencia_df["mes"] == 11).sum())
print("Productos únicos tras preprocesamiento:", inferencia_df["producto_id"].nunique())

# Filtrar solo noviembre y guardar
inferencia_df = inferencia_df[inferencia_df["mes"] == 11].reset_index(drop=True)

output_dir = base_dir.parent / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "inferencia_df_transformado.csv"
inferencia_df.to_csv(output_path, index=False)

print("Transformación completada. Shape final:", inferencia_df.shape)
print("Productos únicos:", inferencia_df["producto_id"].nunique())
print("Archivo guardado en:", output_path)



Registros totales tras preprocesamiento: 888
Registros octubre: 168
Registros noviembre: 720
Productos únicos tras preprocesamiento: 24
Transformación completada. Shape final: (720, 184)
Productos únicos: 24
Archivo guardado en: c:\Users\diego\OneDrive\Desktop\Data_Science\data\processed\inferencia_df_transformado.csv


C:\Users\diego\AppData\Local\Temp\ipykernel_15800\1289041140.py:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  inferencia_df[col] = 0
C:\Users\diego\AppData\Local\Temp\ipykernel_15800\1289041140.py:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  inferencia_df[col] = 0
C:\Users\diego\AppData\Local\Temp\ipykernel_15800\1289041140.py:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using

In [5]:
inferencia_df

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h_Esterilla Yoga.2,subcategoria_h_Mancuernas Ajustables.2,subcategoria_h_Mochila Trekking.2,subcategoria_h_Pesa Rusa.2,subcategoria_h_Pesas Casa.2,subcategoria_h_Rodillera Yoga.2,subcategoria_h_Ropa Montaña.2,subcategoria_h_Ropa Running.2,subcategoria_h_Zapatillas Running.2,subcategoria_h_Zapatillas Trail.2
0,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,0,0,0,0,0,0,0,0,0,0
1,2025-11-01,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,NaN,135.00,NaN,...,0,0,0,0,0,0,0,0,0,0
2,2025-11-01,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,NaN,86.39,NaN,...,0,0,0,0,0,0,0,0,0,0
3,2025-11-01,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,NaN,74.09,NaN,...,0,0,0,0,0,0,0,0,0,0
4,2025-11-01,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,NaN,34.76,NaN,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
715,2025-11-30,PROD_020,Quechua MH500,Outdoor,Ropa Montaña,80,False,NaN,79.64,NaN,...,0,0,0,0,0,0,0,0,0,0
716,2025-11-30,PROD_021,Manduka PRO Yoga Mat,Wellness,Esterilla Yoga,130,True,NaN,130.00,NaN,...,0,0,0,0,0,0,0,0,0,0
717,2025-11-30,PROD_022,Gaiam Premium Yoga Block,Wellness,Bloque Yoga,20,False,NaN,20.18,NaN,...,0,0,0,0,0,0,0,0,0,0
718,2025-11-30,PROD_023,Liforme Yoga Pad,Wellness,Rodillera Yoga,35,False,NaN,34.79,NaN,...,0,0,0,0,0,0,0,0,0,0


In [6]:
inferencia_df.shape

(720, 184)

In [7]:
print(inferencia_df.columns.tolist())

['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria', 'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta', 'ingresos', 'anio', 'mes', 'dia_semana', 'dia_semana_num', 'dia_semana_ordenado', 'es_black_friday', 'dia_mes', 'semana_del_año', 'dia_del_año', 'es_fin_de_semana', 'es_inicio_mes', 'es_fin_mes', 'es_inicio_trimestre', 'es_fin_trimestre', 'es_primera_quincena', 'es_segunda_quincena', 'estacion', 'es_festivo', 'es_ciber_monday', 'es_navidad', 'es_reyes', 'es_san_juan', 'es_verano', 'es_nuevo_año', 'semana_del_anio', 'trimestre', 'cuatrimestre', 'mes_nombre', 'nombre_festivo', 'es_ha_duplicado', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'rolling_mean_7', 'descuento_pct', 'descuento_porcentaje', 'precio_competencia', 'ratio_precio', 'nombre_h_Adidas Own The Run Jacket', 'nombre_h_Adidas Ultraboost 23', 'nombre_h_Asics Gel Nimbus 25', 'nombre_h_Bowflex SelectTech 552', 'nombre_h_Columbia Silver Ridge', 'nombre_h_Decathlon Bandas Elásticas S